# PRE_05 — Anonimización

`data/raw.csv` tiene 600 clientes con identificadores directos (nombre, cédula,
correo, tarjeta de fidelidad) y cuasi-identificadores (edad, ciudad, ocupación).

Borrar los identificadores directos **no alcanza**: la combinación
(edad, ciudad, ocupación) es única en 556 de los 600 registros. `data/auxiliary.csv`
es justo esa información con nombre y apellido — es decir, el ataque de
re-identificación es real y directo.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
import pandas as pd
from PRE_05_anonimizacion.src.anonymize import (
    INPUT_FILE, AUXILIARY_FILE, QUASI_IDENTIFIERS,
    anonymize, generalize, drop_direct_identifiers,
    k_anonymity, reidentification_risk, main,
)

raw = pd.read_csv(INPUT_FILE)
raw.head()

,name,document_id,email,loyalty_card_number,age,city,occupation,annual_spend
0,Camila González,1000000001,camila.gonzalez1@gmail.example,840000000001,25,Envigado,Comerciante,2580000
1,Mateo Ospina,1000000002,mateo.ospina2@yahoo.example,840000000002,36,Cali,Ingeniero de sistemas,5400000
2,Diana Duque,1000000003,diana.duque3@outlook.example,840000000003,38,Bogotá,Docente,5320000
3,Sebastián Álvarez,1000000004,sebastian.alvarez4@yahoo.example,840000000004,67,Itagüí,Abogada,4290000
4,Mariana Henao,1000000005,mariana.henao5@outlook.example,840000000005,29,Bogotá,Comerciante,3670000


## El riesgo antes de anonimizar

Cuántos registros son únicos por sus cuasi-identificadores.

In [2]:
sizes = raw.groupby(["age", "city", "occupation"]).size()
print("combinaciones:", len(sizes))
print("únicas (k=1):", (sizes == 1).sum(), "de", len(raw), "registros")

auxiliary = pd.read_csv(AUXILIARY_FILE)
auxiliary.head()

combinaciones: 578
únicas (k=1): 556 de 600 registros


,name,age,city,occupation
0,Laura Gómez,47,Rionegro,Arquitecta
1,Andrés Restrepo,38,Envigado,Ingeniero de sistemas
2,Camila Henao,29,Pereira,Diseñadora gráfica
3,Carlos Muñoz,61,Bello,Técnico electricista
4,Natalia Ospina,34,Manizales,Docente


## Estrategia

1. Eliminar los identificadores directos.
2. Generalizar los cuasi-identificadores: edad a rangos, ciudad a región, ocupación a sector.
3. Suprimir los grupos que aun así queden con menos de `k` personas.

El resultado cumple k-anonimato: cada registro es indistinguible de al menos otros k-1.

In [3]:
anonymized = main()
anonymized.head(10)

registros: 600 -> 443 (73.8% conservado)
k-anonimato alcanzado: 5 (objetivo 5)
riesgo de re-identificacion: {'personas': 20, 'identificadas': 0, 'sin_coincidencia': 7, 'minimo_candidatos': 5}


,age_range,region,sector,spend_range
0,20-34,Antioquia,Comercio,2-4M
1,20-34,Bogotá D.C.,Comercio,2-4M
2,20-34,Antioquia,Comercio,2-4M
3,35-49,Antioquia,Diseño y construcción,2-4M
4,50+,Bogotá D.C.,Educación,2-4M
5,50+,Antioquia,Salud,4-6M
6,20-34,Antioquia,Diseño y construcción,2-4M
7,50+,Antioquia,Administración y finanzas,6-8M
8,35-49,Antioquia,Comercio,4-6M
9,35-49,Bogotá D.C.,Tecnología,8M+


## Elección de k y del ancho de los rangos de edad

Hay un intercambio directo entre privacidad y utilidad. Con bandas de 10 años
y k=5 se pierde el 44% de los registros; con tres bandas la pérdida baja al 26%
sin bajar el k.

In [4]:
import PRE_05_anonimizacion.src.anonymize as an

configs = {
    "5 bandas": ([0,29,39,49,59,200], ["20-29","30-39","40-49","50-59","60+"]),
    "3 bandas (elegida)": ([0,34,49,200], ["20-34","35-49","50+"]),
    "2 bandas": ([0,44,200], ["20-44","45+"]),
}
original_bins, original_labels = an.AGE_BINS, an.AGE_LABELS
filas = []
for nombre, (bins, labels) in configs.items():
    an.AGE_BINS, an.AGE_LABELS = bins, labels
    for k in (3, 5):
        df = an.anonymize(k=k)
        filas.append({
            "config": nombre, "k objetivo": k,
            "filas": len(df), "conservado": f"{len(df)/len(raw):.1%}",
            "k real": an.k_anonymity(df),
            "identificadas": an.reidentification_risk(df)["identificadas"],
        })
an.AGE_BINS, an.AGE_LABELS = original_bins, original_labels
pd.DataFrame(filas)

,config,k objetivo,filas,conservado,k real,identificadas
0,5 bandas,3,469,78.2%,3,0
1,5 bandas,5,334,55.7%,5,0
2,3 bandas (elegida),3,538,89.7%,3,0
3,3 bandas (elegida),5,443,73.8%,5,0
4,2 bandas,3,577,96.2%,3,0
5,2 bandas,5,520,86.7%,5,0


## Verificación del ataque

Se generalizan los 20 nombres de `auxiliary.csv` igual que la tabla publicada
y se mira a cuántos registros corresponde cada persona. Ninguna debe quedar sola.

In [5]:
reidentification_risk(anonymized)

{'personas': 20,
 'identificadas': 0,
 'sin_coincidencia': 7,
 'minimo_candidatos': 5}

`identificadas: 0` — ninguna de las 20 personas del archivo auxiliar queda
identificada de forma única. `minimo_candidatos: 5` es el peor caso: la persona
más expuesta se confunde con otras 4.